In [1]:
# # %%
# import os
# from dotenv import load_dotenv
# import requests
# import urllib3

# urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# load_dotenv()
# AZURE_OPENAI_EMB_ENDPOINT = os.getenv("AZURE_OPENAI_EMB_ENDPOINT")
# AZURE_OPENAI_EMB_API_KEY = os.getenv("AZURE_OPENAI_EMB_API_KEY")
# AZURE_OPENAI_EMB_API_VERSION = os.getenv("AZURE_OPENAI_EMB_API_VERSION")
# AZURE_OPENAI_EMB_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMB_DEPLOYMENT")

In [ ]:
# # %%
# def get_embedding(text: str):
#     url = f"{AZURE_OPENAI_EMB_ENDPOINT}/openai/deployments/{AZURE_OPENAI_EMB_DEPLOYMENT}/embeddings?api-version={AZURE_OPENAI_EMB_API_VERSION}"

#     headers = {
#         "Content-Type": "application/json",
#         "api-key": AZURE_OPENAI_EMB_API_KEY
#     }

#     payload = {
#         "input": text
#     }

#     response = requests.post(url, headers=headers, json=payload, verify=False)

#     if response.status_code != 200:
#         raise Exception(f"Error: {response.status_code}, {response.text}")

#     result = response.json()

#     # Extract embedding vector
#     embedding = result["data"][0]["embedding"]

#     return embedding


In [5]:
# %%
# -------------------------------
# 0. SET PROJECT ROOT (IMPORTANT)
# -------------------------------
import sys
import os
from weaviate.classes.config import Configure, Property
from weaviate.classes.data import DataObject
import uuid

# PROJECT_ROOT = r"C:\Users\Jevinkumar.Palabhai\OneDrive - TVS Motor Company Ltd\Code-Space\G-RAG\proj-grag"

# PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
# sys.path.append(PROJECT_ROOT)
# if PROJECT_ROOT not in sys.path:



# -------------------------------
# 1. IMPORT MODULES
# -------------------------------
from services.ingestion.pdf_parser import extract_pages
from services.ingestion.block_extractor import extract_blocks
from services.ingestion.chunker import chunk_blocks, build_chunk_text

from services.embedding.embeddings import get_embedding
from services.graph.graph_pipeline import build_graph_from_chunks

from services.vector_db.weaviate_client import WeaviateDB


# -------------------------------
# 2. INPUT
# -------------------------------
# Define a dictionary or list of PDFs with unique Document IDs
PDF_COLLECTION = {
    "DOC_001": r"C:\Users\Dell\OneDrive - Indian Institute of Science\Internship Preparation Roadmap for CDS.pdf",
    "DOC_002": r"C:\Users\Dell\OneDrive - Indian Institute of Science\Academics\LAB\ArjunanSir-CPSdep\Federated Foundation Models on Heterogeneous Time Series (FFTS).pdf",
    "DOC_003": r"C:\Users\Dell\OneDrive - Indian Institute of Science\Academics\LAB\ArjunanSir-CPSdep\FeDaL.pdf",
}

db = WeaviateDB()
print("Initializing schema...")
db.create_schema(reset=False) # Only set True if you want to wipe the DB clean!

# -------------------------------
# 3. MULTI-DOCUMENT INGESTION LOOP
# -------------------------------
for doc_id, pdf_path in PDF_COLLECTION.items():
    print(f"\n{'='*50}\nProcessing {doc_id}: {pdf_path}\n{'='*50}")

    print("Step 1: Extracting pages...")
    pages = extract_pages(pdf_path)

    print("\nStep 2: Extracting blocks...")
    blocks = extract_blocks(pages)

    print("\nStep 3: Chunking...")
    chunks = chunk_blocks(blocks, max_tokens=300, overlap_tokens=50)

    print("\nStep 4: Building chunk texts...")
    chunk_texts = [build_chunk_text(c) for c in chunks if build_chunk_text(c).strip()]
    print(f"Valid chunk_texts: {len(chunk_texts)}")

    print("\nStep 5: Storing Embeddings in Weaviate...")
    chunk_ids = []
    failed_chunks = []

    for i, text in enumerate(chunk_texts):
        try:
            emb = get_embedding(text)
            cid = str(uuid.uuid4())
            chunk_ids.append(cid)

            db.insert_chunk(
                chunk_id=cid,
                text=text,
                embedding=emb,
                doc_id=doc_id  # NEW: Pass the doc_id to Weaviate
            )
        except Exception as e:
            failed_chunks.append(i)

    print(f"Stored {len(chunk_ids)} embeddings. Failed: {len(failed_chunks)}")

    print("\nStep 6: Building Graph in Neo4j...")
    # NEW: Pass the doc_id to the graph builder
    build_graph_from_chunks(chunk_texts, doc_id)

print("\nAll PDFs processed successfully.")


<frozen importlib._bootstrap>:228: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:228: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:228: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


Initializing schema...

Processing DOC_001: C:\Users\Dell\OneDrive - Indian Institute of Science\Internship Preparation Roadmap for CDS.pdf
Step 1: Extracting pages...

Step 2: Extracting blocks...

Step 3: Chunking...

Step 4: Building chunk texts...
Valid chunk_texts: 15

Step 5: Storing Embeddings in Weaviate...


c:\Users\Dell\OneDrive - Indian Institute of Science\Academics\Projects\hybrid-RAG\hybrid-rag-v1\lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'tvsmaznpiaoiprd01.openai.azure.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Stored 15 embeddings. Failed: 0

Step 6: Building Graph in Neo4j...
Processing chunk 1/15


c:\Users\Dell\OneDrive - Indian Institute of Science\Academics\Projects\hybrid-RAG\hybrid-rag-v1\lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'tvsmaznpiaoiprd02.cognitiveservices.azure.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Processing chunk 2/15
Processing chunk 3/15
Processing chunk 4/15
Processing chunk 5/15
Processing chunk 6/15
Processing chunk 7/15
Processing chunk 8/15
Processing chunk 9/15
Processing chunk 10/15
Processing chunk 11/15
Processing chunk 12/15
Processing chunk 13/15
Processing chunk 14/15
Processing chunk 15/15

Processing DOC_002: C:\Users\Dell\OneDrive - Indian Institute of Science\Academics\LAB\ArjunanSir-CPSdep\Federated Foundation Models on Heterogeneous Time Series (FFTS).pdf
Step 1: Extracting pages...

Step 2: Extracting blocks...

Step 3: Chunking...

Step 4: Building chunk texts...
Valid chunk_texts: 32

Step 5: Storing Embeddings in Weaviate...
Stored 32 embeddings. Failed: 0

Step 6: Building Graph in Neo4j...
Processing chunk 1/32
Processing chunk 2/32
Processing chunk 3/32
Processing chunk 4/32
Processing chunk 5/32
Processing chunk 6/32
Processing chunk 7/32
Processing chunk 8/32
Processing chunk 9/32
Processing chunk 10/32
Processing chunk 11/32
Processing chunk 12/32


In [6]:
from services.retrieval.hybrid_engine import HybridQueryEngine

engine = HybridQueryEngine()

questions = [
    "What percentage of Deep Learning preparation is recommended to be completed by the end of December?",
    "Why does the cross-domain fusing approach not work as effectively for time series foundation models as it does for text and images?",
    "Which programming language tends to be commonly asked about during software engineering interviews for systems roles?",
    "What is the title of the 2021 paper authored by H. Zhou that focuses on long sequence time-series forecasting?",
    "What prerequisite must a student obtain before participating in the internship process, and why is it necessary?",
    "Which specific institutions are the authors of the FFTS paper associated with?",
    "Summarize the complete timeline and step-by-step daily routines prescribed for a student preparing for AI/ML/Systems internships during December and January.",
    "Explain how the FFTS framework addresses the challenge of cross-domain statistical heterogeneity in time series foundation model training.",
    
    "What does FeDaL stand for in this paper?",
    "Which machine learning paradigm does this paper use to rethink the development of TSFMs?",
    "What are the three key types of bias identified by the authors that complicate time series generalization?",
    "How does FeDaL specifically mitigate local and global biases within the TSFM architecture?",
    "According to the paper, what are the primary advantages of using a federated learning architecture for training TSFMs?",
    "Explain the server-side core-set tuning mechanism introduced in the paper.",
    "What does the paper's analysis of federated scaling behavior reveal about the relationship between data volume, client count, and model performance?",
    "How does the paper differentiate its approach from existing federated learning works in time series?",
]

for q in questions:
    print(f"(Q): {q}")
    result = engine.query(q)
    print(f"(A): {result['answer']}")
    print("-"*80)
    

(Q): What percentage of Deep Learning preparation is recommended to be completed by the end of December?

[DEBUG] Processed Query Text: 'What percentage of Deep Learning preparation is recommended to be completed by the end of December?'
[DEBUG] Extracted Entities: ['deep learning preparation']
[DEBUG] Vector Store returned: 5 chunks
[DEBUG] Graph Retriever returned: 0 chunks
(A): 50-60% of Deep Learning preparation is recommended to be completed by the end of December.
--------------------------------------------------------------------------------
(Q): Why does the cross-domain fusing approach not work as effectively for time series foundation models as it does for text and images?

[DEBUG] Processed Query Text: 'Why does the cross-domain fusing approach not work as effectively for time series foundation models as it does for text and images?'
[DEBUG] Extracted Entities: ['cross-domain fusing', 'time series foundation models', 'text', 'images']
[DEBUG] Vector Store returned: 5 chunks

In [7]:
# # %%

# from services.retrieval.hybrid_engine import HybridQueryEngine

# engine = HybridQueryEngine()
# fl = False


# while fl:
#     q = input("Enter your query (or 'exit' to quit): ")
#     if q.lower() == 'exit':
#         print("Exiting...")
#         print("-" * 50)
#         fl = False
#         continue
#     result = engine.query(q)
#     print("query=",q)
#     print("result=",result['answer'])
#     print("-" * 50)




## Question bank:

Q) What percentage of Deep Learning preparation is recommended to be completed by the end of December?
Expected Answer: 50-60%

Question 2: "Why does the cross-domain fusing approach not work as effectively for time series foundation models as it does for text and images?"
Expected Answer: Because of significant statistical heterogeneity across domains.

Question 3: "Which programming language tends to be commonly asked about during software engineering interviews for systems roles?"
Expected Answer: C++.


Question 4: "What is the title of the 2021 paper authored by H. Zhou that focuses on long sequence time-series forecasting?"
Expected Answer: Informer: Beyond efficient transformer for long sequence time-series forecasting.


Question 5: "What prerequisite must a student obtain before participating in the internship process, and why is it necessary?"
Expected Answer: Consent from their faculty advisor/guide is required because not all professors permit internships.


Question 6: "Which specific institutions are the authors of the FFTS paper associated with?"
Expected Answer: The Australian Artificial Intelligence Institute at the University of Technology Sydney, and the Department of Data Science and Artificial Intelligence at The Hong Kong Polytechnic University.


Question 7: "Summarize the complete timeline and step-by-step daily routines prescribed for a student preparing for AI/ML/Systems internships during December and January."
Expected Answer: In December, the student must complete ML preparation and 50-60% of Deep Learning while dedicating an hour daily to solve 2-3 DSA questions for 45-50 days. By the end of December or early January, they receive an email from the placement cell. In January, the student must attend lectures, work on projects, and prepare for interviews simultaneously.


Question 8: "Explain how the FFTS framework addresses the challenge of cross-domain statistical heterogeneity in time series foundation model training."
Expected Answer: FFTS treats each data-holding organization as an independent client in a collaborative learning framework with federated settings. It trains multiple client-specific local models to preserve the unique characteristics of each dataset and applies a new regularization mechanism to both the client-side and server-side setups.

Quesiton 9:What is the primary motivation for proposing the FeDaL approach?
Answer: FeDaL aims to address dataset-wise heterogeneity in Time Series Foundation Models (TSFMs), which introduces significant domain biases and degrades model generalization.



Quesiton 10: "What does "FeDaL" stand for in this paper?"
Answer: FeDaL stands for Federated Dataset Learning.  

Quesiton 11: "Which machine learning paradigm does this paper use to rethink the development of TSFMs?"
Answer: The paper rethinks the development of TSFMs using the paradigm of federated learning. 

Quesiton 12: "What are the three key types of bias identified by the authors that complicate time series generalization?"
Answer: The authors identify three key biases: (1) Temporal resolution bias, (2) Physical constraint bias, and (3) Pattern transition bias.  

Quesiton 13: "How does FeDaL specifically mitigate local and global biases within the TSFM architecture?"
Answer: FeDaL explicitly mitigates these biases by adding two complementary mechanisms: Domain Bias Elimination (DBE) and Global Bias Elimination (GBE).  

Quesiton 14: "According to the paper, what are the primary advantages of using a federated learning architecture for training TSFMs?"
Answer: Federated learning allows for collaborative training across distributed data while preserving privacy and reducing the computational burden associated with centralized training.  

Quesiton 15: "Explain the server-side core-set tuning mechanism introduced in the paper."
Answer: To mitigate global bias, the authors introduce a server-side fine-tuning stage using compact core-sets constructed locally by clients. These core-sets approximate the client's data distribution via gradient matching, selecting a small subset whose gradients resemble those of the full dataset.  

Quesiton 16: "What does the paper's analysis of federated scaling behavior reveal about the relationship between data volume, client count, and model performance?"
Answer: The analysis shows that increasing data size consistently improves downstream performance [snippet_14]; furthermore, using more clients (while maintaining a constant total data size) leads to better representations, and higher client participation rates yield stronger results due to more effective aggregation [snippet_14].

Quesiton 17: "How does the paper differentiate its approach from existing federated learning works in time series?"
Answer: While existing works [9] primarily focus on aligning trends across domains, the authors argue these works overlook deeper, dataset-specific structural biases—such as temporal resolution and physical constraint biases—that hinder generalization. FeDaL addresses these by learning dataset-agnostic temporal representations.  

Quesiton 18:
Quesiton 19:

In [8]:
# from services.vector_db.weaviate_client import WeaviateDB

# db = WeaviateDB()
# if db.client.collections.exists("Chunk"):
#     print("Hakunamatata!")
#     print("Dropping Weaviate 'Chunk' collection...")
#     db.client.collections.delete("Chunk")
# print("Weaviate cleared successfully.")